# Digit Recognizer

## Baseline

An introductory computer vision experiment for Kaggle’s **Digit Recognizer** competition: classify a handwritten digit from its image into one of ten classes, **0–9**.

This report includes the notebook’s saved execution outputs: interactive data exploration, CNN training logs, validation metrics, and examples of confidently misclassified digits. No Python environment is required to read the exported HTML report.

### Competition at a glance

| Item | Description |
| --- | --- |
| Task | Multiclass classification of handwritten digits |
| Dataset | MNIST images supplied as CSV rows |
| Image format | 28 × 28 grayscale pixels; 784 pixel values from 0 to 255 |
| Training data | 42,000 labeled images |
| Competition test data | 28,000 images with hidden labels |
| Competition metric | Classification accuracy: the fraction of correctly predicted labels |
| Submission format | A CSV with `ImageId` and `Label` columns |

**Competition links**

- [Overview and evaluation](https://www.kaggle.com/competitions/digit-recognizer/overview)
- [Data description and downloads](https://www.kaggle.com/competitions/digit-recognizer/data)
- [Leaderboard](https://www.kaggle.com/competitions/digit-recognizer/leaderboard)
- [Competition rules](https://www.kaggle.com/competitions/digit-recognizer/rules)

### What this baseline does

1. **Explore the data.**
   - Check the digit distribution and inspect example images with Plotly.
2. **Train a small convolutional neural network.**
   - Normalize pixel values to the range 0–1.
   - Split the labeled data into 80% training and 20% validation, stratified by digit, with seed 42.
   - Train for five epochs using the competition training data only.
3. **Evaluate the predictions.**
   - Compare training and validation loss.
   - Inspect accuracy, precision, recall, F1, and the confusion matrix.
   - Use the legends to compare macro-averaged scores with individual digits.
4. **Inspect model failures and generate a submission.**
   - Review the ten most confident validation errors, with true/predicted labels and their probabilities.
   - Predict labels for the competition test images and save `submission.csv`.

### How to read the results

- **Validation scores** come from the held-out 20% of the labeled training data. They are separate from the Kaggle leaderboard score on the competition test set.
- **Overall (macro)** precision, recall, and F1 give each digit equal weight.
- **Interactive charts** support hovering and legend selection. Click a legend entry to toggle a series; double-click to isolate it.
- **Error-gallery probabilities** are model softmax scores, not calibrated confidence estimates.
- **Execution context** is printed in the environment cell below. Local and remote GPU runs may produce slightly different results.

---

## Before running

1. **Select the GPU kernel.**
   - Local: use the Dev Container's `/usr/bin/python3` kernel.
   - Remote: connect to the active Kaggle Jupyter server and select its Python kernel.
   - Run the environment cell first; check the host, interpreter, GPU, and CUDA result.

2. **Prepare competition access.**
   - Accept the Digit Recognizer competition terms if required.
   - Local: configure `KAGGLE_API_TOKEN` in the repository-root `.env`.
   - Remote: enable a GPU accelerator; KaggleHub uses the Kaggle session to access data.

3. **Run the remaining cells in order.**
   - Inspect the training logs, validation accuracy, and confusion matrix.
   - Review the generated `submission.csv`; this notebook does not submit it to Kaggle.

## Results and reproducibility

- **Model:** a small CNN trained for five epochs with a stratified 80/20 split and seed 42.
- **Environment:** use the supplied scientific packages; versions are printed below.
- **Outputs:** local files persist in the mounted repository; remote files remain on the Kaggle server.
- **Git:** working notebook outputs are preserved, while the Git clean filter strips committed outputs and execution counts.
- **Reports:** export HTML after both local and remote execution have been verified.


In [ ]:
import os
import sys
import time
import random
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

# fix seed to reproduce results

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

# resource check
print("Host:", os.uname().nodename)
print("Python:", sys.version)
print("Interpreter:", sys.executable)

assert torch.cuda.is_available()
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

print({p: version(p) for p in ["torch", "numpy", "pandas", "scikit-learn", "plotly"]})

probe = torch.arange(16, dtype=torch.float32, device=device)
assert (probe @ probe).item() == 1240.0
print("CUDA computation passed")

# Embed Plotly assets so notebook outputs can be viewed without a CDN.
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"


## Data preparation

### Input files

- **Existing project data**
  - Reuse `train.csv` and `test.csv` when both are present in `competitions/digit-recognizer/data/`.
- **Data resolved by KaggleHub**
  - Otherwise, call `kagglehub.competition_download()` and read from its returned directory.
  - Local: download into `competitions/digit-recognizer/data/` in the mounted repository.
  - Kaggle: use the data location resolved by KaggleHub; the requested download directory may be ignored.

### Authentication

- **Local**
  - Set `KAGGLE_API_TOKEN` in the repository-root `.env`, using `.env.example` as a template.
  - `load_dotenv()` loads the file; KaggleHub reads the token from the environment.
  - Existing environment variables take precedence. Restart the kernel after changing the token.
- **Remote**
  - Use the Kaggle session's authentication. The remote kernel cannot read the local `.env`.
- **Credentials**
  - Keep `.env` ignored by Git. Never include tokens or Jupyter connection URLs in notebook cells or outputs.

### Generated files

- **From the local repository:** `competitions/digit-recognizer/output/`
- **From the competition directory:** `output/`
- **From Kaggle's default working directory:** `/kaggle/working/digit-recognizer/output/`

Local data and generated files are ignored by Git. Copy remote results back to the repository when needed.


In [ ]:
import kagglehub
from dotenv import load_dotenv

# Find the repository from its root, competition folder, or a nested directory.
repo_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "competitions" / "digit-recognizer").is_dir()
    ),
    None,
)
project_dir = (
    repo_dir / "competitions" / "digit-recognizer"
    if repo_dir is not None
    else (
        Path.cwd()
        if Path.cwd().name == "digit-recognizer"
        else Path.cwd() / "digit-recognizer"
    )
)
output_dir = project_dir / "output"

if repo_dir is not None:
    load_dotenv(repo_dir / ".env", override=False)

# KaggleHub downloads locally and resolves mounted data on Kaggle.
# Always read from its returned path; Kaggle may ignore output_dir.
data_dir = project_dir / "data"
if not all((data_dir / name).is_file() for name in ("train.csv", "test.csv")):
    data_dir = Path(
        kagglehub.competition_download(
            "digit-recognizer",
            output_dir=str(data_dir.resolve()),
        )
    )

if not all((data_dir / name).is_file() for name in ("train.csv", "test.csv")):
    raise FileNotFoundError(f"Check the competition CSV files: {data_dir}")

print("Competition data:", data_dir.resolve())
print("Output directory:", output_dir.resolve())


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

assert "label" in train.columns
pixels = [f"pixel{i}" for i in range(784)]

assert list(train.drop(columns="label").columns) == pixels
assert list(test.columns) == pixels
assert train["label"].between(0, 9).all()
assert not train.isna().any().any() and not test.isna().any().any()
assert train[pixels].min().min() >= 0 and train[pixels].max().max() <= 255
assert test.min().min() >= 0 and test.max().max() <= 255

print("Data:", data_dir.resolve())
print("Shapes:", train.shape, test.shape)
display(train.head())


## Exploratory data analysis

- Check the class distribution.
- Inspect one training image for each digit.

In [ ]:
class_counts = train["label"].value_counts().sort_index()
fig = px.bar(
    x=class_counts.index.astype(str),
    y=class_counts.values,
    labels={"x": "Digit", "y": "Training samples"},
    title="Training label distribution",
    text_auto=True,
)
fig.show()


def digit_gallery(images, titles, title, captions_below=False):
    count = len(images)
    columns = 5
    rows = (count + columns - 1) // columns
    fig = make_subplots(
        rows=rows,
        cols=columns,
        subplot_titles=None if captions_below else titles,
        vertical_spacing=0.30 if captions_below else 0.15,
    )
    for index, image in enumerate(images):
        fig.add_trace(
            go.Heatmap(
                z=image.reshape(28, 28),
                colorscale="Gray",
                zmin=0,
                zmax=1,
                showscale=False,
                hovertemplate="x=%{x}, y=%{y}<br>Intensity=%{z:.2f}<extra></extra>",
            ),
            row=index // columns + 1,
            col=index % columns + 1,
        )
    if captions_below:
        for index, caption in enumerate(titles):
            fig.add_annotation(
                text=caption,
                x=0.5,
                y=0,
                xref="x domain",
                yref="y domain",
                xanchor="center",
                yanchor="top",
                yshift=-10,
                showarrow=False,
                align="center",
                font=dict(size=12),
                row=index // columns + 1,
                col=index % columns + 1,
            )

    fig.update_xaxes(visible=False, constrain="domain")
    for index in range(rows * columns):
        axis = "x" if index == 0 else f"x{index + 1}"
        fig.update_yaxes(
            visible=False,
            autorange="reversed",
            scaleanchor=axis,
            scaleratio=1,
            row=index // columns + 1,
            col=index % columns + 1,
        )
    fig.update_layout(
        title=title,
        height=rows * (280 if captions_below else 230) + 100,
        margin=dict(t=80 if captions_below else 110, b=90 if captions_below else 40),
    )
    return fig


example_indices = [train.index[train["label"] == digit][0] for digit in range(10)]
digit_gallery(
    train.loc[example_indices, pixels].to_numpy(dtype=np.float32) / 255.0,
    [f"Label: {digit}" for digit in range(10)],
    "One training example per digit",
).show()


## Model training

Train the CNN for five epochs. Record training and validation cross-entropy loss, validation accuracy, and macro-averaged precision, recall, and F1 after each epoch.

Macro averaging gives each digit equal weight; undefined precision or recall is reported as zero.

In [ ]:
X = train[pixels].to_numpy(dtype=np.float32) / 255.0
y = train["label"].to_numpy(dtype=np.int64)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)


def loader(x, y, shuffle=False):
    return DataLoader(
        TensorDataset(torch.from_numpy(x).reshape(-1, 1, 28, 28), torch.from_numpy(y)),
        batch_size=256,
        shuffle=shuffle,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )


train_loader = loader(X_train, y_train, True)
valid_loader = loader(X_valid, y_valid)

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
history = []
metric_history = []

for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    model.eval()
    validation_loss = 0.0
    probability_batches = []
    with torch.inference_mode():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            validation_loss += criterion(logits, labels).item() * len(labels)
            probability_batches.append(logits.softmax(dim=1).cpu().numpy())

    valid_probabilities = np.concatenate(probability_batches)
    predictions = valid_probabilities.argmax(axis=1)
    class_precision, class_recall, class_f1, _ = precision_recall_fscore_support(
        y_valid, predictions, labels=np.arange(10), zero_division=0
    )
    precision = class_precision.mean()
    recall = class_recall.mean()
    f1 = class_f1.mean()

    metric_history.append(
        dict(
            epoch=epoch,
            target="Overall (macro)",
            precision=precision,
            recall=recall,
            f1=f1,
        )
    )
    for digit in range(10):
        metric_history.append(
            dict(
                epoch=epoch,
                target=f"Digit {digit}",
                precision=class_precision[digit],
                recall=class_recall[digit],
                f1=class_f1[digit],
            )
        )
    row = dict(
        epoch=epoch,
        loss=total_loss / len(X_train),
        validation_loss=validation_loss / len(X_valid),
        accuracy=accuracy_score(y_valid, predictions),
        precision=precision,
        recall=recall,
        f1=f1,
        seconds=time.perf_counter() - started,
    )
    history.append(row)
    print(row)

display(pd.DataFrame(history))


## Learning curves

- Compare training and validation loss, then inspect overall validation accuracy.
- Precision, recall, and F1 share a legend across three panels.
  - **Overall (macro):** the unweighted mean across the ten digits; shown initially.
  - **Digit 0–9:** click a legend entry to show or hide that digit in all three panels.
  - Double-click an entry to isolate it; double-click again to restore the other series.
- All scores are computed on the validation split after each epoch. Per-digit precision, recall, and F1 treat that digit as the positive class.


In [ ]:
history_df = pd.DataFrame(history)
assert np.isfinite(history_df.select_dtypes(include="number")).all().all()

loss_data = history_df.melt(
    id_vars="epoch",
    value_vars=["loss", "validation_loss"],
    var_name="Split",
    value_name="Cross-entropy loss",
)
fig = px.line(
    loss_data,
    x="epoch",
    y="Cross-entropy loss",
    color="Split",
    markers=True,
    title="Training and validation loss",
)
fig.update_xaxes(dtick=1, title="Epoch")
fig.show()

fig = px.line(
    history_df,
    x="epoch",
    y="accuracy",
    markers=True,
    title="Overall validation accuracy",
)
fig.update_xaxes(dtick=1, title="Epoch")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

metric_history_df = pd.DataFrame(metric_history)
metric_names = {"precision": "Precision", "recall": "Recall", "f1": "F1"}
targets = ["Overall (macro)"] + [f"Digit {i}" for i in range(10)]
colors = ["#111827"] + px.colors.qualitative.Plotly

fig = make_subplots(
    rows=1, cols=3, subplot_titles=list(metric_names.values()), shared_yaxes=True
)
for target_index, target in enumerate(targets):
    rows = metric_history_df[metric_history_df["target"] == target]
    for column, (metric, label) in enumerate(metric_names.items(), start=1):
        fig.add_trace(
            go.Scatter(
                x=rows["epoch"],
                y=rows[metric],
                mode="lines+markers",
                name=target,
                legendgroup=target,
                showlegend=column == 1,
                visible=True if target_index == 0 else "legendonly",
                line=dict(
                    color=colors[target_index], width=3 if target_index == 0 else 2
                ),
                hovertemplate=f"{target}<br>Epoch %{{x}}<br>{label}: %{{y:.3%}}<extra></extra>",
            ),
            row=1,
            col=column,
        )
fig.update_xaxes(dtick=1, title="Epoch")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.update_layout(
    title="Validation precision, recall, and F1 by epoch",
    legend=dict(
        title="Click to toggle; double-click to isolate", groupclick="togglegroup"
    ),
    height=450,
)
fig.show()


## Validation by digit

- The confusion matrix uses true labels on rows and predicted labels on columns.
- The table lists each digit’s final precision, recall, F1, and support.
- The final metric chart uses the same overall/digit legend controls as the learning curves.
  - The overall macro average is shown initially.
  - Enable individual digits to compare them with the overall score.


In [ ]:
assert len(predictions) == len(y_valid)
cm = confusion_matrix(y_valid, predictions, labels=np.arange(10))
fig = px.imshow(
    cm,
    x=[str(i) for i in range(10)],
    y=[str(i) for i in range(10)],
    text_auto=True,
    color_continuous_scale="Blues",
    labels=dict(x="Predicted digit", y="True digit", color="Samples"),
    title="Validation confusion matrix",
)
fig.show()

precision, recall, f1, support = precision_recall_fscore_support(
    y_valid,
    predictions,
    labels=np.arange(10),
    zero_division=0,
)
class_metrics = pd.DataFrame(
    {
        "digit": np.arange(10).astype(str),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    }
)
display(class_metrics)
final_metrics = metric_history_df[metric_history_df["epoch"] == EPOCHS]
final_metric_data = final_metrics.melt(
    id_vars=["epoch", "target"],
    value_vars=list(metric_names),
    var_name="Metric",
    value_name="Score",
)
fig = px.bar(
    final_metric_data,
    x="Metric",
    y="Score",
    color="target",
    barmode="group",
    category_orders={"target": targets},
    color_discrete_map=dict(zip(targets, colors)),
    title="Final validation metrics: overall and per digit",
    labels={"target": "Overall / digit"},
)
for trace in fig.data:
    trace.visible = True if trace.name == "Overall (macro)" else "legendonly"
fig.update_layout(legend=dict(title="Click to toggle; double-click to isolate"))
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()


## Generate the competition submission

Predict test labels and save the submission and training history. No Kaggle submission is sent automatically.

In [ ]:
test_x = test.to_numpy(dtype=np.float32) / 255.0
model.eval()
with torch.inference_mode():
    test_predictions = np.concatenate(
        [
            model(
                torch.from_numpy(test_x[i : i + 256]).reshape(-1, 1, 28, 28).to(device)
            )
            .argmax(1)
            .cpu()
            .numpy()
            for i in range(0, len(test_x), 256)
        ]
    )
submission = pd.DataFrame(
    {"ImageId": np.arange(1, len(test_predictions) + 1), "Label": test_predictions}
)
assert len(submission) == len(test) and submission["Label"].between(0, 9).all()

submission.to_csv(output_dir / "submission.csv", index=False)
metric_history_df.to_csv(output_dir / "validation_metrics_by_digit.csv", index=False)
pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)

print("Saved:", output_dir / "submission.csv")
display(submission.head())


## Inspect the ten most confident misclassifications

- Select up to ten validation errors with the highest predicted-label softmax probabilities, in descending order (left to right, then top to bottom).
- Beneath each digit image, show two lines:
  - `True: 3 (0.01%)`
  - `Pred: 2 (99.99%)`
- The true-label and predicted-label probabilities come from the same prediction. They need not sum to 100% because there are ten possible digits.
- These softmax scores are not calibrated confidence estimates. Inspect the strokes to understand confusion; some errors may still look obvious to a person.


In [ ]:
wrong_indices = np.flatnonzero(predictions != y_valid)
confidence = valid_probabilities.max(axis=1)
# Show confidently wrong predictions rather than selecting examples by appearance.
selected = wrong_indices[np.argsort(-confidence[wrong_indices], kind="stable")[:10]]

if len(selected) == 0:
    print("No misclassified validation samples.")
else:
    error_examples = pd.DataFrame(
        {
            "validation_position": selected,
            "true_label": y_valid[selected],
            "predicted_label": predictions[selected],
            "predicted_probability": confidence[selected],
            "true_label_probability": valid_probabilities[selected, y_valid[selected]],
        }
    )
    display(error_examples)
    titles = [
        f"True: {y_valid[i]} ({valid_probabilities[i, y_valid[i]]:.2%})<br>"
        f"Pred: {predictions[i]} ({confidence[i]:.2%})"
        for i in selected
    ]
    fig = digit_gallery(
        X_valid[selected],
        titles,
        "Top 10 misclassified validation samples by predicted probability",
        captions_below=True,
    )
    fig.update_annotations(font_size=12)

    for trace, i in zip(fig.data, selected):
        trace.hovertemplate = (
            f"True: {y_valid[i]}<br>"
            f"Predicted: {predictions[i]}<br>"
            f"P(predicted): {confidence[i]:.2%}<br>"
            f"P(true): {valid_probabilities[i, y_valid[i]]:.2%}"
            "<extra></extra>"
        )

    fig.show()
